## DLC Analysis


In [ ]:
# Load all the tools
import os

import numpy as np

from openfast_io import FileTools

import pandas as pd


import matplotlib.pyplot as plt

import openmdao.api as om

import matplotlib
# font = {
# #         'family' : 'normal',
# #         'weight' : 'bold',
#         'size'   : 14
#        }

# matplotlib.rc('font', **font)

import ruamel.yaml as ry


Timeseries plotting function

Pass a dict of `{label: timeseries}` (or a list of DataFrames) to `plot_tss` to overlay multiple cases on the same axes.

In [ ]:
def plot_tss(dfs, channels, labels=None, legend=True, **plot_kwargs):
    '''
    Plot timeseries channels, overlaying one or more cases on the same axes.

    Parameters
    ----------
    dfs : DataFrame, list of DataFrames, or dict of {label: DataFrame}
        Timeseries to plot.  If a dict is given, its keys are used as the legend labels.
    channels : list of str
        Channels to plot, one subplot per channel.
    labels : list of str, optional
        Legend label for each case.  Ignored if `dfs` is a dict.
    legend : bool
        Draw a legend when more than one case is plotted.
    plot_kwargs
        Passed through to `ax.plot()`.

    Returns
    -------
    fig, axs
    '''

    if isinstance(dfs, dict):
        if labels is None:
            labels = list(dfs.keys())
        dfs = list(dfs.values())
    elif isinstance(dfs, pd.DataFrame):
        dfs = [dfs]
    else:
        dfs = list(dfs)

    if labels is None:
        labels = [f'Case {i_df}' for i_df in range(len(dfs))]
    elif isinstance(labels, str):
        labels = [labels]

    if len(labels) != len(dfs):
        raise ValueError(f'Got {len(labels)} labels for {len(dfs)} cases')

    fig, axs = plt.subplots(len(channels), 1)
    fig.set_size_inches(12, 2 * len(channels))

    axs = np.atleast_1d(axs).flatten()

    for df, label in zip(dfs, labels):
        for i_chan, chan in enumerate(channels):
            if chan not in df:
                continue
            axs[i_chan].plot(df.Time, df[chan], label=label, **plot_kwargs)

    for i_chan, chan in enumerate(channels):
        axs[i_chan].set_ylabel(chan)
        axs[i_chan].grid(True)

    axs[-1].set_xlabel('Time')

    [a.set_xticklabels('') for a in axs[:-1]]

    if legend and len(dfs) > 1:
        axs[0].legend(
            ncol=min(len(dfs), 4),
            loc='lower center',
            bbox_to_anchor=(0.5, 1.02),
            frameon=False,
        )

    fig.patch.set_facecolor('white')
    fig.align_ylabels()

    return fig, axs

# Case Info

In [ ]:
def get_weis_paths(weis_output_dir, run_folder='openfast_runs', rank=0, iteration=0, use_mpi=None):
    '''
    Construct paths into a WEIS output directory.

    Only `weis_output_dir` needs to be set for most cases: e.g.
    /scratch/dzalkind/RM1-MHK/0_setup_dlcs

    Parameters
    ----------
    weis_output_dir : str
        Top-level WEIS output folder (what you point WEIS's `general.folder_output` at)
    run_folder : str
        Name of the folder containing the OpenFAST run outputs (default 'openfast_runs')
    rank : int
        MPI rank folder to use, if WEIS was run with MPI (default 0)
    iteration : int
        Optimization iteration to use. Always 0 for a single DLC run (default 0)
    use_mpi : bool, optional
        Whether a rank_X subfolder is present. If None (default), this is auto-detected
        based on whether the rank_{rank} folder exists

    Returns
    -------
    dict with keys: run_dir, case_matrix, iteration_dir, summary_stats, dels, timeseries_dir
    '''
    run_dir = os.path.join(weis_output_dir, run_folder)

    if use_mpi is None:
        use_mpi = os.path.isdir(os.path.join(run_dir, f'rank_{rank}'))

    if use_mpi:
        run_dir = os.path.join(run_dir, f'rank_{rank}')

    iteration_dir = os.path.join(run_dir, f'iteration_{iteration}')

    return {
        'run_dir': run_dir,
        'case_matrix': os.path.join(run_dir, 'case_matrix_combined.yaml'),
        'iteration_dir': iteration_dir,
        'summary_stats': os.path.join(iteration_dir, 'summary_stats.p'),
        'dels': os.path.join(iteration_dir, 'DELs.p'),
        'timeseries_dir': os.path.join(iteration_dir, 'timeseries'),
    }


In [ ]:

# Function for reading case matrix
def read_cm(fname_case_matrix):
    cm_dict = FileTools.load_yaml(fname_case_matrix, package=1)
    cnames = []
    for c in list(cm_dict.keys()):
        if isinstance(c,ry.comments.CommentedKeySeq):
            cnames.append(tuple(c))
        else:
            cnames.append(c)

    cm = pd.DataFrame(cm_dict, columns=cnames)

    cm['DLC'].unique()

    dlc_inds = {}

    for dlc in cm['DLC'].unique():
        dlc_inds[dlc] = cm['DLC'] == dlc
        
    return cm, dlc_inds


In [ ]:
# Point this at any WEIS output folder; the openfast_runs/rank_0/iteration_0
# subfolders are resolved automatically by get_weis_paths()
weis_output_dir = '/scratch/dzalkind/RM1-MHK/9_full_trans/baseline'

weis_paths = get_weis_paths(weis_output_dir)

cm, dlc_inds = read_cm(weis_paths['case_matrix'])

cm


# Summary Stats
Generated if
        `save_timeseries: True`,
        `save_iterations: True`
in modeling options

Includes stats for all the OpenFAST outputs, if the simulations run to completion

In [ ]:
ss = pd.read_pickle(weis_paths['summary_stats'])
ss


In [ ]:
## Detect and exclude failed simulations
# WEIS/OpenFAST fills every channel with -9999 and flags 'openfast_failed' when a run fails
if 'openfast_failed' in ss.columns.get_level_values(0):
    failed = ss['openfast_failed']['mean'] > 0
else:
    # Fallback for older runs without the 'openfast_failed' channel
    failed = (ss.xs('max', axis=1, level=1) == -9999).all(axis=1)

print(f"{failed.sum()} of {len(failed)} simulations failed")
if failed.any():
    print(ss.index[failed].tolist())

# Drop failed runs from summary stats and case matrix so later cells only see valid data
ss = ss.loc[~failed]
cm = cm.loc[~failed.values].reset_index(drop=True)

dlc_inds = {}
for dlc in cm['DLC'].unique():
    dlc_inds[dlc] = cm['DLC'] == dlc


In [ ]:
ss['GenSpeed']['max'].sort_values(ascending=False).head(10)

ss['PtfmPitch']['max'].sort_values(ascending=True).head(20)


In [ ]:
def mag_approx(df,chans,mag_chan='shear'):
    df[mag_chan] = np.sqrt(np.sum(df[chans]**2,axis=1))
    
    
def max_df(df,chans,max_or_min='max'):
    max_df = pd.DataFrame()
    for chan in chans:
        max_df[chan] = df[chan][max_or_min]
        
    return max_df
    

In [ ]:
# Script for checking min/max of each dlc

dlcs = ['6.1b']
# dlcs = ['5.1']
channels = ['PtfmSurge', 'PtfmSway', 'PtfmHeave', 'PtfmRoll','PtfmPitch','PtfmYaw']


for dlc in dlcs:
    for chan in channels:
        ss_max = ss.reset_index()[dlc_inds[dlc].values][chan]['max']
        ss_min = ss.reset_index()[dlc_inds[dlc].values][chan]['min']
        i_max = ss_max.argmax()
        i_min = ss_min.argmin()
        print(f"DLC {dlc}: Max {chan} of {ss_max.max():.3f} in {ss.index[ss_max.index[i_max]]}")
        print(f"DLC {dlc}: Min {chan} of {ss_min.max():.3f} in {ss.index[ss_min.index[i_min]]}")


# ss.index[2]

# Plot maxima for each DLC vs. wind speed

In [ ]:
dlcs = cm['DLC'].unique()
dlc_labs = [f'DLC {num}' for num in dlcs]


channels = ['GenSpeed','PtfmPitch','PtfmRoll','PtfmYaw']


# fig, axs = plt.subplots(len(channels),1)
fig, axs = plt.subplots(2,2)
axs = axs.flatten()
fig.set_size_inches(16,3*len(channels))



for i_chan, chan in enumerate(channels):

    for i_dlc, dlc in enumerate(dlcs):

        dlc_ind = cm['DLC'] == dlc
        axs[i_chan].scatter(
            ss.reset_index()['Wind1VelX']['mean'].to_numpy()[dlc_ind],
            ss.reset_index()[chan]['max'].to_numpy()[dlc_ind],
            label=dlc_labs[i_dlc]
        )
        axs[i_chan].grid()
        axs[i_chan].set_xlabel('Mean Wind Speed (m/s)')
        axs[i_chan].set_ylabel(f'Max {chan}')
        axs[i_chan].grid(True)


# axs[0].plot([0,30],[1.3*7.56,1.3*7.56],'k--')
# [a.set_xlim([3,28]) for a in axs]

fig.legend(
    [f'DLC {d}' for d in dlcs],
    ncol=3,
    loc='upper right', bbox_to_anchor=(.675, .93)
)

# [a.grid() for a in axs]

# Plotting

In [ ]:

# List the cases to overlay.  Add/remove entries to compare more or fewer runs.
case_names = [
    # 'DLC4.1_5_weis_job_1',
    # 'DLC6.1a_7_weis_job_0',
    # 'DLC6.1b_8_weis_job_0',
    # 'DLC6.2_9_weis_job_0',
    'DLCAEP_2_weis_job_0',
    'DLCAEP_2_weis_job_1',
    'DLCAEP_2_weis_job_2',
    'DLCAEP_2_weis_job_3',
    # 'DLC7.2_10_weis_job_1',
    # 'DLC5.1_6_weis_job_1',
    # 'DLC5.1_6_weis_job_2',
    # 'DLC1.1_0_weis_job_1',
    # 'DLC1.1_0_weis_job_2',
]

tss = {
    case_name: pd.read_pickle(os.path.join(weis_paths['timeseries_dir'], f'{case_name}.p'))
    for case_name in case_names
}

ts1 = tss[case_names[0]]

channels = ['Wind1VelX','RtVAvgxh','GenTq','BldPitch1','RotSpeed','PtfmPitch','PtfmHeave','PtfmRoll','PtfmYaw','NacYaw','TwrBsMyt','RootMyb1','Wave1Elev']
fig,axs = plot_tss(tss,channels)

# [a.set_xlim([200,300]) for a in axs]


In [ ]:
ts1['Wind1VelX'].std() / ts1['Wind1VelX'].mean()